# 04 — Multiplicación de Matrices

**Objetivo:** Entender por qué la multiplicación matricial no es elemento a elemento, su interpretación geométrica como composición de transformaciones, y aplicaciones directas en atribución de canales.

## Contexto matemático

Si $A \in \mathbb{R}^{m \times k}$ y $B \in \mathbb{R}^{k \times n}$, entonces:

$$C = AB \in \mathbb{R}^{m \times n}, \quad c_{ij} = \sum_{l=1}^{k} a_{il} b_{lj}$$

**Regla del dedo:** las dimensiones **internas** deben coincidir ($k$). Las dimensiones **externas** forman el resultado ($m \times n$).

$$\underbrace{(m \times k)}_{A} \cdot \underbrace{(k \times n)}_{B} = \underbrace{(m \times n)}_{C}$$

In [ ]:
import os
os.chdir('/Volumes/SSD_Gabo/proyectos/growth-analytics')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams.update({
    'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e8e8e8', 'axes.axisbelow': True,
    'axes.titlesize': 11, 'axes.titleweight': 'bold', 'legend.frameon': False,
})
np.random.seed(42)
print('Setup listo')

## 1 — La regla de la dimensión y por qué no es elemento a elemento

In [ ]:
A = np.array([[1, 2, 3],
              [4, 5, 6]])   # shape (2, 3)

B = np.array([[7, 8],
              [9, 10],
              [11, 12]])    # shape (3, 2)

C = A @ B   # (2×3) @ (3×2) = (2×2)
print(f'A: {A.shape}, B: {B.shape}, C=A@B: {C.shape}')
print('C:')
print(C)

# Verificación manual de c[0,0]
c00_manual = np.dot(A[0], B[:, 0])
print(f'\nc[0,0] = A[fila0] · B[col0] = {A[0]} · {B[:,0]} = {c00_manual}')

# NO conmutativos
try:
    print('B @ A:', (B @ A).shape, '  ← distinta shape que A@B')
    print('A@B ≠ B@A (ni siquiera misma shape aquí)')
except Exception as e:
    print(e)

## 2 — Propiedades

| Propiedad | Fórmula | Válida |
|---|---|---|
| Asociatividad | $(AB)C = A(BC)$ | ✓ Siempre |
| Distributividad | $A(B+C) = AB + AC$ | ✓ Siempre |
| Conmutatividad | $AB = BA$ | ✗ En general NO |
| Traspuesta de producto | $(AB)^T = B^T A^T$ | ✓ Siempre |

**Nota importante:** $AB = BA$ puede ser verdad para matrices específicas (ej: $A$ y $A^{-1}$, matrices diagonales), pero no en general.

In [ ]:
P = np.random.randint(1, 5, (3, 3)).astype(float)
Q = np.random.randint(1, 5, (3, 3)).astype(float)
R = np.random.randint(1, 5, (3, 3)).astype(float)

print('Asociatividad (PQ)R = P(QR):', np.allclose((P@Q)@R, P@(Q@R)))
print('Distributividad P(Q+R) = PQ+PR:', np.allclose(P@(Q+R), P@Q+P@R))
print('NO conmutativa PQ == QP:', np.allclose(P@Q, Q@P))
print('(PQ)^T = Q^T P^T:', np.allclose((P@Q).T, Q.T @ P.T))

## 3 — Multiplicación como combinación lineal de columnas

El producto $A\mathbf{x}$ puede interpretarse como **combinación lineal de las columnas de $A$** con coeficientes $x_i$:

$$A\mathbf{x} = x_1 \mathbf{a}_1 + x_2 \mathbf{a}_2 + \cdots + x_n \mathbf{a}_n$$

donde $\mathbf{a}_j$ es la $j$-ésima columna de $A$. Esto conecta multiplicación matricial con el concepto de **espacio columna**.

In [ ]:
# Escenario de atribución: mix de 3 presupuestos para 4 canales
# S: escenarios × canales  (3 escenarios, 4 canales)
# CVR: canales × 1         (tasa de conversión por canal)
# Resultado: escenarios × 1 (activaciones esperadas por escenario)

S = np.array([
    [5000, 3000, 2000, 1000],   # Escenario A: foco paid
    [2000, 2000, 4000, 2000],   # Escenario B: balanceado
    [1000, 1000, 1000, 8000],   # Escenario C: foco referral
])

CVR = np.array([0.032, 0.041, 0.058, 0.075])   # tasas de conversión por canal

activaciones = S @ CVR
print('Activaciones esperadas por escenario:')
for i, (esc, act) in enumerate(zip(['A','B','C'], activaciones)):
    print(f'  Escenario {esc}: {act:.1f} activaciones')

# Verificación combinación lineal para escenario A
comb_lineal = sum(S[0, j] * CVR[j] for j in range(4))
print(f'\nCombinación lineal manual escenario A: {comb_lineal:.1f}')
print(f'Resultado con @: {activaciones[0]:.1f}')

## 4 — Potencias de matriz y efectos compuestos

Si $T$ es una **matriz de transición** (probabilidades), $T^k$ representa aplicar la transición $k$ veces. Ejemplo: cadena de Markov del funnel de usuario.

In [ ]:
# Matriz de transición del funnel (probabilidades de avanzar entre etapas)
# Estados: [Visitante, Registrado, Activado, Pagante, Churned]
T = np.array([
    [0.40, 0.35, 0.00, 0.00, 0.25],  # Visitante
    [0.00, 0.50, 0.30, 0.00, 0.20],  # Registrado
    [0.00, 0.00, 0.45, 0.40, 0.15],  # Activado
    [0.00, 0.00, 0.00, 0.85, 0.15],  # Pagante
    [0.00, 0.00, 0.00, 0.00, 1.00],  # Churned
])

# Estado inicial: 1000 visitantes nuevos
estado_0 = np.array([1000, 0, 0, 0, 0], dtype=float)

estados = [estado_0]
for k in range(1, 7):
    estados.append(np.linalg.matrix_power(T.T, k) @ estado_0)

print('Evolución del funnel (usuarios por estado):')
print(f'{"Semana":>7}  {"Visit":>8}  {"Reg":>8}  {"Act":>8}  {"Pago":>8}  {"Churn":>8}')
etqs = ['Visit','Reg','Act','Pago','Churn']
for k, est in enumerate(estados):
    print(f'{k:>7}  {est[0]:>8.0f}  {est[1]:>8.0f}  {est[2]:>8.0f}  {est[3]:>8.0f}  {est[4]:>8.0f}')

## 5 — Visualización: evolución del funnel de Markov

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
semanas = list(range(7))
colores = ['#4361ee', '#06d6a0', '#ff9f1c', '#7209b7', '#adb5bd']

for i, (etq, col) in enumerate(zip(etqs, colores)):
    vals = [estados[k][i] for k in range(7)]
    ax.plot(semanas, vals, 'o-', color=col, lw=2, ms=6, label=etq)

ax.set_xlabel('Semanas')
ax.set_ylabel('Usuarios en cada estado')
ax.set_title('Evolución del funnel: multiplicación matricial iterada $T^k \\cdot s_0$')
ax.legend()
plt.tight_layout()
plt.show()

## Resumen

| Concepto | Descripción | NumPy |
|---|---|---|
| Multiplicación matricial | $(m\times k)(k\times n) = (m\times n)$ | `A @ B` |
| Producto punto | Caso especial vector-vector | `np.dot(u, v)` |
| NO conmutativa | $AB \neq BA$ en general | — |
| Producto matriz-vector | Combinación lineal de columnas | `A @ x` |
| Potencias de matriz | Aplicar transformación $k$ veces | `np.linalg.matrix_power(A, k)` |

**Siguiente:** `05_determinants.ipynb`